# Análise de Impacto da Integração de Veículos Elétricos em Redes de Distribuição
## Estudo de Caso: Alimentador Real ESB01S4 (Co-simulação Mosaik / OpenDSS / Caldera ICM)
### **Módulo 1: Avaliação de Conformidade Regulatória ANEEL (PRODIST Módulo 8 / REN 1.000/2021) com Desagregação de Distúrbios (Cela et al., 2024 - UFC) no Caso Base (0 VEs)**

Este notebook constitui o ambiente principal de experimentação, análise estatística e geração de figuras do TCC.
Nesta etapa, executamos e diagnosticamos exaustivamente o **Caso Base** (rede de distribuição real operando com 1.914 cargas ativas de Baixa Tensão, cargas de Média Tensão e 303 sistemas fotovoltaicos parametrizados em $\text{FP}=1{,}0$, porém com **0 Veículos Elétricos**).

A simulação é executada com resolução temporal de **10 minutos durante 7 dias consecutivos (1.008 passos temporais)**, reproduzindo rigorosamente a **janela de amostragem semanal de 168 horas** padronizada pela ANEEL no **PRODIST Módulo 8 (Qualidade da Energia Elétrica - QEE)**.

### **Metodologia de Desagregação dos Distúrbios de Tensão (Cela et al., 2024)**
Para superar a limitação do PRODIST Módulo 8 convencional — que agrupa subtensões e sobretensões em índices agregados ($DRP$ e $DRC$), mascarando a real causa física dos distúrbios —, este trabalho incorpora a formulação matemática proposta por pesquisadores da Universidade Federal do Ceará (UFC):

> **Cela, J. A. C. A., Marinho, L. A. S. S., Melo, L. S., Gregory, R. C. F., & Leão, R. P. S. (2024)**. *Ferramenta para Apuração Automática de Indicadores de Tensão em Regime Permanente: Uma Análise Comparativa entre Normas Nacionais e Internacionais*. Departamento de Engenharia Elétrica, Universidade Federal do Ceará (UFC).

As seguintes métricas regulatórias da ANEEL são avaliadas no Caso Base:
1. **Níveis de Tensão em Regime Permanente** (Faixas Adequada, Precária e Crítica);
2. **Indicadores DRP e DRC Desagregados (Cela et al., 2024)**: Subtensão Precária ($\text{DRP}_{\text{sub}}$), Sobretensão Precária ($\text{DRP}_{\text{sob}}$), Subtensão Crítica ($\text{DRC}_{\text{sub}}$) e Sobretensão Crítica ($\text{DRC}_{\text{sob}}$);
3. **Desbalanceamento de Tensão**: Fator de Desbalanceamento Nodal ($FD_{\%}$ / VUF%) e indicador estatístico **$FD_{95\%}$** (Limite $\le 2{,}0\%$);
4. **Fator de Potência (FP)**: Conformidade com a faixa normativa ($0{,}92 \le \text{FP} \le 1{,}00$) e análise de fluxo de reativos;
5. **Capacidade Operativa e Carregamento de Linhas**: Ampacidade e correntes de fase;
6. **Demanda Agregada, Geração Solar e Balanço Energético**: Curva de carga semanal e produção solar (kW e kWh);
7. **Dashboard Consolidado de Conformidade ANEEL**: Tabela resumo com o diagnóstico regulatório completo;
8. **Módulo Comparativo Automatizado**: Função parametrizada para avaliar o impacto das taxas de penetração de VEs (5%, 10%, 20%, 30%).

---

In [ ]:
import importlib
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Habilita autoreload para atualizar automaticamente qualquer arquivo Python alterado em disco
%load_ext autoreload
%autoreload 2

# Localização robusta da raiz do projeto
curr = Path.cwd().resolve()
while curr.parent != curr:
    if (curr / "pyproject.toml").exists():
        PROJECT_ROOT = curr
        break
    curr = curr.parent
else:
    PROJECT_ROOT = Path("..").resolve()

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Configuração de estilo visual acadêmico para gráficos do TCC
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "grid.alpha": 0.4,
    "grid.linestyle": "--",
})

# Força o recarregamento do despachante Mosaik
import scenarios.scenario_ESB01S4_TCC
importlib.reload(scenarios.scenario_ESB01S4_TCC)
from scenarios.scenario_ESB01S4_TCC import run_scenario

print("Ambiente configurado com sucesso!")
print(f"Diretório raiz do projeto: {PROJECT_ROOT}")

## 1. Execução / Carregamento do Caso Base (Baseline - 0 VEs)

O Caso Base corresponde ao alimentador real operando com todas as cargas residenciais de Baixa Tensão (BT) saneadas (1.914 ativas), cargas de Média Tensão (MT), 303 sistemas fotovoltaicos (PVs em `conn=Delta`, $\text{FP}=1{,}0$) e reguladores de tensão ativos, com **0 Veículos Elétricos**.

Para executar a co-simulação completa (7 dias com passos de 10 minutos = 1.008 passos), configure `EXECUTAR_SIMULACAO = True`.

In [ ]:
ARQUIVO_BASELINE = PROJECT_ROOT / "output" / "result_TCC_baseline.csv"

# Defina EXECUTAR_SIMULACAO = True apenas se desejar re-executar a simulação Mosaik do zero
EXECUTAR_SIMULACAO = False

if EXECUTAR_SIMULACAO or not ARQUIVO_BASELINE.exists():
    print("Iniciando co-simulação do Caso Base (0 VEs) via Mosaik...")
    arquivo_gerado = run_scenario(
        qtd_evs=0,
        penetracao_pct=0.0,
        usar_pv_mosaik=False,
        nome_cenario="result_TCC_baseline",
    )
    print(f"Simulação concluída! Resultados gravados em: {ARQUIVO_BASELINE}")
else:
    print(f"Carregando resultados prévios de: {ARQUIVO_BASELINE.name}")

df_base = pd.read_csv(ARQUIVO_BASELINE)
df_base['date'] = pd.to_datetime(df_base['date'])
df_base.set_index('date', inplace=True)

print(f"\nResumo dos dados carregados:")
print(f"  - Período: {df_base.index.min()} até {df_base.index.max()}")
print(f"  - Total de passos temporais: {len(df_base)}")
print(f"  - Total de variáveis coletadas: {df_base.shape[1]}")

## 2. Mapeamento e Categorização das Grandezas Monitoradas

In [ ]:
# Mapeamento das colunas do dataset
colunas_v1 = [c for c in df_base.columns if "V1_pu" in c]
colunas_v2 = [c for c in df_base.columns if "V2_pu" in c]
colunas_v3 = [c for c in df_base.columns if "V3_pu" in c]
colunas_i_line = [c for c in df_base.columns if "Line" in c and any(i in c for i in ["I1_A", "I2_A", "I3_A"])]
colunas_p_line = [c for c in df_base.columns if "Line" in c and any(p in c for p in ["P1_w", "P2_w", "P3_w"])]
colunas_q_line = [c for c in df_base.columns if "Line" in c and any(q in c for q in ["Q1_var", "Q2_var", "Q3_var"])]
colunas_pvs = [c for c in df_base.columns if "PVSystem" in c and "P_meas" in c]

print(f"Categorias de sensores identificadas:")
print(f"  - Barras BT com medição Fase A (V1): {len(colunas_v1)}")
print(f"  - Barras BT com medição Fase B (V2): {len(colunas_v2)}")
print(f"  - Barras BT com medição Fase C (V3): {len(colunas_v3)}")
print(f"  - Sensores de Corrente de Linha: {len(colunas_i_line)}")
print(f"  - Sensores de Potência Ativa/Reativa de Linha: {len(colunas_p_line)} / {len(colunas_q_line)}")
print(f"  - Geradores Solares Monitorados (PVs): {len(colunas_pvs)}")

## 3. MÉTRICA ANEEL 1: Níveis de Tensão em Regime Permanente e Indicadores DRP / DRC
### **Metodologia PRODIST Módulo 8 Desagregada (Cela et al., 2024 - UFC)**

De acordo com o **Módulo 8 do PRODIST**, a tensão em regime permanente em Baixa Tensão ($V_{\text{nom}} = 220\text{ V} / 380\text{ V}$) é classificada nas seguintes faixas:
- **Tensão Adequada:** $0{,}92 \le V_{\text{pu}} \le 1{,}05$
- **Tensão Precária:** $0{,}87 \le V_{\text{pu}} < 0{,}92$ ou $1{,}05 < V_{\text{pu}} \le 1{,}06$
- **Tensão Crítica:** $V_{\text{pu}} < 0{,}87$ ou $V_{\text{pu}} > 1{,}06$

### **Desagregação de Subtensão e Sobretensão**
O PRODIST convencional calcula os indicadores agregados $\text{DRP} \le 3{,}0\%$ e $\text{DRC} \le 0{,}5\%$. Conforme demonstrado por **Cela et al. (2024)**, esses indicadores agregados perdem a informação sobre a origem do distúrbio. Para superar essa limitação, desagregamos os índices em:

$$\text{DRP} = \text{DRP}_{\text{sub}} + \text{DRP}_{\text{sob}}$$
$$\text{DRC} = \text{DRC}_{\text{sub}} + \text{DRC}_{\text{sob}}$$

Onde:
- $\text{DRP}_{\text{sub}}$: Subtensão Precária ($0{,}87 \le V < 0{,}92\text{ pu}$)
- $\text{DRP}_{\text{sob}}$: Sobretensão Precária ($1{,}05 < V \le 1{,}06\text{ pu}$)
- $\text{DRC}_{\text{sub}}$: Subtensão Crítica ($V < 0{,}87\text{ pu}$)
- $\text{DRC}_{\text{sob}}$: Sobretensão Crítica ($V > 1{,}06\text{ pu}$)

In [ ]:
# Filtragem de valores válidos de tensão (> 0.1 pu)
def filtrar_tensoes(df, cols):
    df_sub = df[cols].replace([np.inf, -np.inf], np.nan)
    return df_sub[df_sub > 0.1]

df_v1 = filtrar_tensoes(df_base, colunas_v1)
df_v2 = filtrar_tensoes(df_base, colunas_v2)
df_v3 = filtrar_tensoes(df_base, colunas_v3)

todas_tensoes = pd.concat([df_v1.stack(), df_v2.stack(), df_v3.stack()])
total_medicoes = len(todas_tensoes)
n_passos = len(df_base)

# Função para decomposição vetorial do PRODIST segundo Cela et al. (2024)
def decompor_prodist(df_fase):
    m_prec_sub = (df_fase >= 0.87) & (df_fase < 0.92)
    m_prec_sob = (df_fase > 1.05) & (df_fase <= 1.06)
    m_crit_sub = df_fase < 0.87
    m_crit_sob = df_fase > 1.06
    
    drp_sub = (m_prec_sub.sum(axis=0) / n_passos) * 100.0
    drp_sob = (m_prec_sob.sum(axis=0) / n_passos) * 100.0
    drc_sub = (m_crit_sub.sum(axis=0) / n_passos) * 100.0
    drc_sob = (m_crit_sob.sum(axis=0) / n_passos) * 100.0
    
    drp_total = drp_sub + drp_sob
    drc_total = drc_sub + drc_sob
    
    return pd.DataFrame({
        "DRP (%)": drp_total,
        "DRP_sub (%)": drp_sub,
        "DRP_sob (%)": drp_sob,
        "DRC (%)": drc_total,
        "DRC_sub (%)": drc_sub,
        "DRC_sob (%)": drc_sob
    })

dec_v1 = decompor_prodist(df_v1)
dec_v2 = decompor_prodist(df_v2)
dec_v3 = decompor_prodist(df_v3)

# Tabela comparativa decomposta por fase (formato Cela et al., 2024)
tabela_prodist_decomposta = pd.DataFrame([
    {"Fase": "Fase A", "DRP (%)": dec_v1["DRP (%)"].mean(), "DRP_sub (%)": dec_v1["DRP_sub (%)"].mean(), "DRP_sob (%)": dec_v1["DRP_sob (%)"].mean(),
     "DRC (%)": dec_v1["DRC (%)"].mean(), "DRC_sub (%)": dec_v1["DRC_sub (%)"].mean(), "DRC_sob (%)": dec_v1["DRC_sob (%)"].mean()},
    {"Fase": "Fase B", "DRP (%)": dec_v2["DRP (%)"].mean(), "DRP_sub (%)": dec_v2["DRP_sub (%)"].mean(), "DRP_sob (%)": dec_v2["DRP_sob (%)"].mean(),
     "DRC (%)": dec_v2["DRC (%)"].mean(), "DRC_sub (%)": dec_v2["DRC_sub (%)"].mean(), "DRC_sob (%)": dec_v2["DRC_sob (%)"].mean()},
    {"Fase": "Fase C", "DRP (%)": dec_v3["DRP (%)"].mean(), "DRP_sub (%)": dec_v3["DRP_sub (%)"].mean(), "DRP_sob (%)": dec_v3["DRP_sob (%)"].mean(),
     "DRC (%)": dec_v3["DRC (%)"].mean(), "DRC_sub (%)": dec_v3["DRC_sub (%)"].mean(), "DRC_sob (%)": dec_v3["DRC_sob (%)"].mean()},
]).set_index("Fase")

print("=" * 85)
print(" INDICADORES PRODIST MÓDULO 8 DECOMPOSTOS POR FASE — METODOLOGIA CELA ET AL. (2024)")
print("=" * 85)
display(tabela_prodist_decomposta.round(4))

# Avaliação de Conformidade Global
pct_adequada = 100.0 * np.sum((todas_tensoes >= 0.92) & (todas_tensoes <= 1.05)) / total_medicoes
pct_precaria = 100.0 * np.sum(((todas_tensoes >= 0.87) & (todas_tensoes < 0.92)) | ((todas_tensoes > 1.05) & (todas_tensoes <= 1.06))) / total_medicoes
pct_critica = 100.0 * np.sum((todas_tensoes < 0.87) | (todas_tensoes > 1.06)) / total_medicoes

# Identificação de barras não conformes (DRP > 3.0% e DRC > 0.5%)
todas_barras_dec = pd.concat([dec_v1, dec_v2, dec_v3])
barras_violam_drp = todas_barras_dec[todas_barras_dec["DRP (%)"] > 3.0]
barras_violam_drc = todas_barras_dec[todas_barras_dec["DRC (%)"] > 0.5]

print(f"\nDiagnóstico Global das Medições:")
print(f"  - Tensão Média Global:   {todas_tensoes.mean():.4f} pu")
print(f"  - Tensão Mínima Global:  {todas_tensoes.min():.4f} pu")
print(f"  - Tensão Máxima Global:  {todas_tensoes.max():.4f} pu")
print(f"  - Faixa Adequada (0.92 a 1.05 pu):    {pct_adequada:.2f}%")
print(f"  - Faixa Precária (0.87 a 0.92 pu):    {pct_precaria:.2f}%")
print(f"  - Faixa Crítica  (< 0.87 ou > 1.06 pu): {pct_critica:.2f}%")
print(f"\nConformidade dos Indicadores Regulatórios ANEEL (Limites: DRP <= 3.0%, DRC <= 0.5%):")
print(f"  - Barras com violação de DRP (> 3.0%): {len(barras_violam_drp)} ({100.0 * len(barras_violam_drp) / len(todas_barras_dec):.2f}%)")
print(f"  - Barras com violação de DRC (> 0.5%): {len(barras_violam_drc)} ({100.0 * len(barras_violam_drc) / len(todas_barras_dec):.2f}%)")
print("=" * 85)

### 3.1. Séries Temporais e Envoltória de Tensão com Faixas PRODIST

In [ ]:
v1_media, v1_min, v1_max = df_v1.mean(axis=1), df_v1.min(axis=1), df_v1.max(axis=1)
v2_media, v2_min, v2_max = df_v2.mean(axis=1), df_v2.min(axis=1), df_v2.max(axis=1)
v3_media, v3_min, v3_max = df_v3.mean(axis=1), df_v3.min(axis=1), df_v3.max(axis=1)
dias = df_base.index

fig, ax = plt.subplots(figsize=(15, 5.5))

# Faixas normativas PRODIST
ax.axhspan(0.92, 1.05, color="green", alpha=0.10, label="Faixa Adequada (0.92 a 1.05 pu)")
ax.axhspan(0.87, 0.92, color="gold", alpha=0.15, label="Faixa Precária Inferior (0.87 a 0.92 pu)")
ax.axhspan(1.05, 1.06, color="gold", alpha=0.15, label="Faixa Precária Superior (1.05 a 1.06 pu)")
ax.axhline(0.87, color="red", linestyle=":", linewidth=1.5, label="Limite Crítico Inferior (0.87 pu)")
ax.axhline(1.06, color="darkred", linestyle=":", linewidth=1.5, label="Limite Crítico Superior (1.06 pu)")

# Curvas médias por fase
ax.plot(dias, v1_media, color="#d9534f", linewidth=1.6, label="Fase A (V1) - Média")
ax.plot(dias, v2_media, color="#5cb85c", linewidth=1.6, label="Fase B (V2) - Média")
ax.plot(dias, v3_media, color="#0275d8", linewidth=1.6, label="Fase C (V3) - Média")

# Envoltória min-max global
v_min_global = pd.concat([v1_min, v2_min, v3_min], axis=1).min(axis=1)
v_max_global = pd.concat([v1_max, v2_max, v3_max], axis=1).max(axis=1)
ax.fill_between(dias, v_min_global, v_max_global, color="gray", alpha=0.18, label="Envoltória Mín-Máx das Barras BT")

ax.set_title("Comportamento da Tensão em Baixa Tensão ao Longo de 7 Dias - Caso Base (0 VEs)", fontweight="bold")
ax.set_ylabel("Tensão Nodal (pu)")
ax.set_xlabel("Dia e Horário")
ax.grid(True)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%a, %d/%m %Hh"))
plt.xticks(rotation=30)
ax.set_ylim(0.80, 1.15)
ax.legend(loc="upper right", ncol=2)
plt.tight_layout()
plt.show()

### 3.2. Dispersão e Distribuição Estatística de Tensão (Boxplot e Histograma de Densidade)

In [ ]:
fig, (ax_box, ax_hist) = plt.subplots(1, 2, figsize=(15, 5))

dados_box = [df_v1.stack().dropna(), df_v2.stack().dropna(), df_v3.stack().dropna()]
ax_box.boxplot(dados_box, tick_labels=["Fase A", "Fase B", "Fase C"], patch_artist=True,
               boxprops=dict(facecolor="#e0e0e0", color="black"),
               medianprops=dict(color="crimson", linewidth=2))
ax_box.axhline(0.92, color="orange", linestyle="--", linewidth=1.2, label="Limite Adequado (0.92)")
ax_box.axhline(1.05, color="darkred", linestyle="--", linewidth=1.2, label="Limite Adequado (1.05)")
ax_box.set_title("Dispersão dos Níveis de Tensão por Fase (Boxplot)", fontweight="bold")
ax_box.set_ylabel("Tensão (pu)")
ax_box.grid(True)
ax_box.legend(loc="upper right")

ax_hist.hist(dados_box[0], bins=80, density=True, alpha=0.55, color="#d9534f", label="Fase A")
ax_hist.hist(dados_box[1], bins=80, density=True, alpha=0.55, color="#5cb85c", label="Fase B")
ax_hist.hist(dados_box[2], bins=80, density=True, alpha=0.55, color="#0275d8", label="Fase C")
ax_hist.axvline(0.92, color="orange", linestyle="--", linewidth=1.5, label="Limite Inferior (0.92)")
ax_hist.axvline(1.05, color="darkred", linestyle="--", linewidth=1.5, label="Limite Superior (1.05)")
ax_hist.axvspan(0.92, 1.05, color="lightgreen", alpha=0.15, label="Faixa Adequada")
ax_hist.set_title("Histograma de Densidade de Tensão (pu)", fontweight="bold")
ax_hist.set_xlabel("Tensão (pu)")
ax_hist.set_ylabel("Densidade de Probabilidade")
ax_hist.grid(True)
ax_hist.legend(loc="upper left")

plt.tight_layout()
plt.show()

### 3.3. Ranking das Barras com Maiores Índices de Transgressão (DRP e DRC)

In [ ]:
# Ranking das 10 barras mais críticas
df_criticas = todas_barras_dec.sort_values(by="DRP (%)", ascending=False)
print("Top 10 Barras com Maiores Transgressões de Tensão no Alimentador:")
display(df_criticas.head(10).round(4))

## 4. MÉTRICA ANEEL 2: Desbalanceamento de Tensão (FD% / VUF% - PRODIST Módulo 8)
O Fator de Desbalanceamento de Tensão ($FD_{\%}$ / VUF%) quantifica a assimetria entre as fases do sistema gerada pela dispersão desequilibrada de cargas monofásicas e bifásicas:

$$FD_{\%, i}(t) \approx \frac{\max\left( |V_{A,i} - V_{\text{médio},i}|, |V_{B,i} - V_{\text{médio},i}|, |V_{C,i} - V_{\text{médio},i}| \right)}{V_{\text{médio},i}} \times 100\%$$

O PRODIST estabelece que o **indicador $FD_{95\%}$** (o valor de desbalanceamento que não é superado em **$95\%$ das 1.008 leituras semanais**) não deve ultrapassar **$2{,}0\%$** em regime permanente.

In [ ]:
# Identifica barras verdadeiramente trifásicas (todas as 3 fases energizadas > 0.5 pu)
# Exclui ramais monofásicos e bifásicos onde fases ausentes registram 0V e distorceriam o VUF
buses_v1_map = {c.replace("-V1_pu", ""): c for c in colunas_v1}
buses_v2_map = {c.replace("-V2_pu", ""): c for c in colunas_v2}
buses_v3_map = {c.replace("-V3_pu", ""): c for c in colunas_v3}
barras_trifasicas_candidatas = sorted(list(set(buses_v1_map.keys()) & set(buses_v2_map.keys()) & set(buses_v3_map.keys())))

v1_init = df_base[[buses_v1_map[b] for b in barras_trifasicas_candidatas]].iloc[0]
v2_init = df_base[[buses_v2_map[b] for b in barras_trifasicas_candidatas]].iloc[0]
v3_init = df_base[[buses_v3_map[b] for b in barras_trifasicas_candidatas]].iloc[0]
barras_trifasicas = [
    b for b in barras_trifasicas_candidatas
    if v1_init[buses_v1_map[b]] > 0.5 and v2_init[buses_v2_map[b]] > 0.5 and v3_init[buses_v3_map[b]] > 0.5
]

cols_v1_c = [buses_v1_map[b] for b in barras_trifasicas]
cols_v2_c = [buses_v2_map[b] for b in barras_trifasicas]
cols_v3_c = [buses_v3_map[b] for b in barras_trifasicas]

v1_mat = df_base[cols_v1_c].values
v2_mat = df_base[cols_v2_c].values
v3_mat = df_base[cols_v3_c].values
vm_mat = (v1_mat + v2_mat + v3_mat) / 3.0

desv_max_mat = np.maximum(np.maximum(np.abs(v1_mat - vm_mat), np.abs(v2_mat - vm_mat)), np.abs(v3_mat - vm_mat))
mask_valida = vm_mat > 0.1
vuf_mat = np.where(mask_valida, (desv_max_mat / vm_mat) * 100.0, np.nan)
df_vuf_nodal = pd.DataFrame(vuf_mat, index=df_base.index, columns=barras_trifasicas)

# Indicador Regulatório FD95% para cada barra
fd95_por_barra = df_vuf_nodal.quantile(0.95, axis=0)
barras_violam_fd95 = fd95_por_barra[fd95_por_barra > 2.0]
pct_conformidade_fd = 100.0 * (1.0 - len(barras_violam_fd95) / len(fd95_por_barra))

vuf_medio_tempo = df_vuf_nodal.mean(axis=1)
vuf_max_tempo = df_vuf_nodal.max(axis=1)

print("=" * 70)
print(" DIAGNÓSTICO DO FATOR DE DESBALANCEAMENTO DE TENSÃO (FD% / VUF%)")
print("=" * 70)
print(f"VUF Médio Global:           {vuf_medio_tempo.mean():.3f}%")
print(f"VUF Máximo Global (Pico):   {vuf_max_tempo.max():.3f}%")
print(f"FD95% Máximo no Alimentador: {fd95_por_barra.max():.3f}%")
print(f"FD95% Médio no Alimentador:  {fd95_por_barra.mean():.3f}%")
print(f"Barras com FD95% > 2.0% (Não Conforme): {len(barras_violam_fd95)} de {len(fd95_por_barra)} ({100.0 - pct_conformidade_fd:.2f}%)")
print(f"Taxa de Conformidade Regulatória (FD95% <= 2.0%): {pct_conformidade_fd:.2f}%")
print("=" * 70)

fig, (ax_temp, ax_dist) = plt.subplots(1, 2, figsize=(15, 4.5))
ax_temp.plot(dias, vuf_medio_tempo, color="purple", linewidth=1.6, label="VUF Médio do Alimentador (%)")
ax_temp.plot(dias, vuf_max_tempo, color="darkviolet", linestyle=":", linewidth=1.2, label="Pico Máximo Nodal (%)")
ax_temp.axhline(2.0, color="red", linestyle="--", linewidth=1.5, label="Limite PRODIST / IEEE (2,0%)")
ax_temp.set_title("VUF Temporal ao Longo da Semana - Caso Base", fontweight="bold")
ax_temp.set_ylabel("Desbalanceamento VUF (%)")
ax_temp.set_xlabel("Dia e Horário")
ax_temp.grid(True)
ax_temp.xaxis.set_major_formatter(mdates.DateFormatter("%a, %d/%m"))
plt.setp(ax_temp.get_xticklabels(), rotation=30)
ax_temp.set_ylim(0, max(2.5, vuf_max_tempo.max() * 1.1))
ax_temp.legend(loc="upper right")

ax_dist.hist(fd95_por_barra.dropna(), bins=40, color="mediumpurple", edgecolor="black", alpha=0.7)
ax_dist.axvline(2.0, color="red", linestyle="--", linewidth=1.5, label="Limite ANEEL (2,0%)")
ax_dist.set_title("Distribuição do Indicador FD95% nas Barras BT", fontweight="bold")
ax_dist.set_xlabel("FD95% da Barra (%)")
ax_dist.set_ylabel("Número de Barras")
ax_dist.grid(True)
ax_dist.legend(loc="upper right")

plt.tight_layout()
plt.show()

## 5. MÉTRICA ANEEL 3: Fator de Potência e Fluxo de Potência Reativa (PRODIST e REN 1.000/2021)
Limite Regulatório da ANEEL: $0{,}92 \le \text{FP} \le 1{,}00$.

In [ ]:
# Identifica a linha tronco principal do alimentador (maior fluxo de corrente/potência)
line_names = sorted(list(set([c.split('-')[2] for c in df_base.columns if 'Line' in c])))
linhas_info = []
for l in line_names:
    i_cols = [c for c in df_base.columns if l in c and any(i in c for i in ['I1_A', 'I2_A', 'I3_A'])]
    p_cols = [c for c in df_base.columns if l in c and any(p in c for p in ['P1_w', 'P2_w', 'P3_w'])]
    q_cols = [c for c in df_base.columns if l in c and any(q in c for q in ['Q1_var', 'Q2_var', 'Q3_var'])]
    if i_cols and p_cols:
        linhas_info.append({'linha': l, 'i_max': df_base[i_cols].max().max(), 'p_cols': p_cols, 'q_cols': q_cols, 'i_cols': i_cols})

df_linhas_rank = pd.DataFrame(linhas_info).sort_values(by='i_max', ascending=False)
linha_tronco = df_linhas_rank.iloc[0]

# Potências no OpenDSS já são reportadas em kW e kvar
p_tronco_kw = df_base[linha_tronco['p_cols']].sum(axis=1).abs()
q_tronco_kvar = df_base[linha_tronco['q_cols']].sum(axis=1).abs()
s_tronco_kva = np.sqrt(p_tronco_kw**2 + q_tronco_kvar**2)

fp_agregado = np.where(s_tronco_kva > 1e-2, p_tronco_kw / s_tronco_kva, 1.0)
fp_agregado = pd.Series(fp_agregado, index=df_base.index)
horas_abaixo_092 = (fp_agregado < 0.92).sum() / len(fp_agregado) * 100.0

print("=" * 75)
print(f" DIAGNÓSTICO DO FATOR DE POTÊNCIA NA LINHA TRONCO ({linha_tronco['linha']})")
print("=" * 75)
print(f"Corrente Máxima da Linha Tronco: {linha_tronco['i_max']:.2f} A")
print(f"Fator de Potência Médio:         {fp_agregado.mean():.4f}")
print(f"Fator de Potência Mínimo:        {fp_agregado.min():.4f}")
print(f"Horas com FP < 0.92:             {horas_abaixo_092:.2f}% (efeito da injeção solar ativa ao meio-dia)")
print("=" * 75)

fig, ax = plt.subplots(figsize=(15, 4.5))
ax.plot(dias, fp_agregado, color="teal", linewidth=1.6, label=f"Fator de Potência Tronco ({linha_tronco['linha']})")
ax.axhline(0.92, color="red", linestyle="--", linewidth=1.5, label="Limite Regulatório ANEEL (0,92)")
ax.axhline(1.00, color="gray", linestyle=":", linewidth=1.0)
ax.set_title("Comportamento do Fator de Potência na Linha Tronco do Alimentador - Caso Base", fontweight="bold")
ax.set_ylabel("Fator de Potência")
ax.set_xlabel("Dia e Horário")
ax.grid(True)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%a, %d/%m"))
plt.xticks(rotation=30)
ax.set_ylim(min(0.15, fp_agregado.min() * 0.95), 1.02)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 6. MÉTRICA ANEEL 4: Carregamento de Linhas e Ampacidade dos Condutores

In [ ]:
if colunas_i_line:
    df_correntes = df_base[colunas_i_line]
    corrente_maxima_tempo = df_correntes.max(axis=1)
    corrente_media_tempo = df_correntes.mean(axis=1)
    
    print("=" * 70)
    print(" DIAGNÓSTICO DE CARREGAMENTO DE CORRENTE NAS LINHAS (CASO BASE)")
    print("=" * 70)
    print(f"Corrente Máxima Absoluta Registrada: {df_correntes.max().max():.2f} A")
    print(f"Corrente Média das Linhas:           {df_correntes.mean().mean():.2f} A")
    print(f"Pico de Corrente Máxima Nodal:       {corrente_maxima_tempo.max():.2f} A")
    print("=" * 70)
    
    fig, ax = plt.subplots(figsize=(15, 4.5))
    ax.plot(dias, corrente_maxima_tempo, color="firebrick", linewidth=1.6, label="Corrente Máxima de Linha (A)")
    ax.plot(dias, corrente_media_tempo, color="orange", linestyle="--", linewidth=1.4, label="Corrente Média Amostrada (A)")
    ax.set_title("Perfil de Carregamento de Corrente das Linhas Monitoradas - Caso Base", fontweight="bold")
    ax.set_ylabel("Corrente (A)")
    ax.set_xlabel("Dia e Horário")
    ax.grid(True)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%a, %d/%m"))
    plt.xticks(rotation=30)
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()
else:
    print("Sensores de corrente de linha não encontrados.")

## 7. MÉTRICA ANEEL 5: Demanda Agregada, Geração Solar e Balanço Energético

In [ ]:
# Demanda ativa da linha tronco e geração solar dos 303 PVs em kW
p_linhas_kw = p_tronco_kw
p_solar_kw = df_base[colunas_pvs].sum(axis=1) if colunas_pvs else pd.Series(0, index=df_base.index)
demanda_liquida_kw = p_linhas_kw - p_solar_kw

delta_t_h = 10.0 / 60.0
energia_linhas_kwh = (p_linhas_kw * delta_t_h).sum()
energia_solar_kwh = (p_solar_kw * delta_t_h).sum()

print("=" * 70)
print(" BALANÇO DE ENERGIA E POTÊNCIA AGREGADA DO ALIMENTADOR (SEMANAL)")
print("=" * 70)
print(f"Pico Máximo de Demanda Tronco:     {p_linhas_kw.max():.2f} kW")
print(f"Pico Máximo de Geração Solar (PVs): {p_solar_kw.max():.2f} kW")
print(f"Demanda Média no Período:          {p_linhas_kw.mean():.2f} kW")
print(f"Energia Total Ativa Amostrada:      {energia_linhas_kwh:,.1f} kWh")
print(f"Energia Solar Total Produzida:      {energia_solar_kwh:,.1f} kWh")
print("=" * 70)

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(dias, p_linhas_kw, color="navy", linewidth=1.6, label="Demanda de Carga nas Linhas (kW)")
ax.plot(dias, p_solar_kw, color="gold", linewidth=1.5, label="Geração Solar dos 303 PVs (kW)")
ax.fill_between(dias, 0, p_solar_kw, color="yellow", alpha=0.25)
ax.set_title("Curva de Carga Agregada vs. Geração Solar Fotovoltaica - Caso Base (0 VEs)", fontweight="bold")
ax.set_ylabel("Potência Ativa (kW)")
ax.set_xlabel("Dia e Horário")
ax.grid(True)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%a, %d/%m"))
plt.xticks(rotation=30)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 8. Dashboard Consolidado de Conformidade Regulatória ANEEL

In [ ]:
status_drp = "CONFORME" if len(barras_violam_drp) == 0 else f"NÃO CONFORME ({len(barras_violam_drp)} barras)"
status_drc = "CONFORME" if len(barras_violam_drc) == 0 else f"NÃO CONFORME ({len(barras_violam_drc)} barras)"
status_fd95 = "CONFORME" if len(barras_violam_fd95) == 0 else f"NÃO CONFORME ({len(barras_violam_fd95)} barras)"
status_fp = "CONFORME" if fp_agregado.min() >= 0.92 else f"ALERTA ({horas_abaixo_092:.1f}% horas < 0.92)"

dashboard_data = [
    {"Métrica Regulatória": "Tensão Adequada (0.92 a 1.05 pu)", "Referência": "PRODIST Módulo 8", "Limite Normativo": "Desejável ~100%", "Valor Observado": f"{pct_adequada:.2f}%", "Status": "MONITORADO"},
    {"Métrica Regulatória": "Tensão Precária (DRP)", "Referência": "PRODIST Módulo 8", "Limite Normativo": "DRP <= 3.0%", "Valor Observado": f"DRP Méd: {tabela_prodist_decomposta['DRP (%)'].mean():.2f}%", "Status": status_drp},
    {"Métrica Regulatória": "Tensão Crítica (DRC)", "Referência": "PRODIST Módulo 8", "Limite Normativo": "DRC <= 0.5%", "Valor Observado": f"DRC Méd: {tabela_prodist_decomposta['DRC (%)'].mean():.2f}%", "Status": status_drc},
    {"Métrica Regulatória": "Desagregação Sub/Sob (DRP)", "Referência": "Cela et al. (2024)", "Limite Normativo": "Diagnóstico Físico", "Valor Observado": f"Sub: {tabela_prodist_decomposta['DRP_sub (%)'].mean():.2f}% | Sob: {tabela_prodist_decomposta['DRP_sob (%)'].mean():.2f}%", "Status": "DIAGNÓSTICO"},
    {"Métrica Regulatória": "Desbalanceamento (FD95%)", "Referência": "PRODIST Módulo 8", "Limite Normativo": "FD95% <= 2.0%", "Valor Observado": f"FD95% Máx: {fd95_por_barra.max():.2f}% (Méd: {fd95_por_barra.mean():.2f}%)", "Status": status_fd95},
    {"Métrica Regulatória": "Fator de Potência Tronco", "Referência": "REN 1.000 / Mód. 8", "Limite Normativo": "0.92 <= FP <= 1.00", "Valor Observado": f"FP Méd: {fp_agregado.mean():.4f} (Mín: {fp_agregado.min():.4f})", "Status": status_fp},
    {"Métrica Regulatória": "Pico Máximo de Demanda Tronco", "Referência": "Capacidade Ativos", "Limite Normativo": "Ampacidade Condutores", "Valor Observado": f"{p_linhas_kw.max():.2f} kW", "Status": "MONITORADO"},
    {"Métrica Regulatória": "Corrente Máxima de Linha", "Referência": "Térmico Linhas", "Limite Normativo": "I_adm Condutor", "Valor Observado": f"{corrente_maxima_tempo.max():.2f} A", "Status": "MONITORADO"},
    {"Métrica Regulatória": "Energia Semanal Entregue Tronco", "Referência": "Balanço Energético", "Limite Normativo": "-", "Valor Observado": f"{energia_linhas_kwh:,.1f} kWh", "Status": "INFO"},
]

df_resumo_aneel = pd.DataFrame(dashboard_data)
print("=" * 85)
print(" QUADRO DE CONFORMIDADE REGULATÓRIA ANEEL - CASO BASE (0 VEs)")
print("=" * 85)
display(df_resumo_aneel)

## 9. Módulo Comparativo Automatizado para Cenários de Penetração de VEs (5%, 10%, 20%, 30%)
Esta função automatizada recebe os arquivos de saída dos cenários com Veículos Elétricos e executa a comparação diferencial direta com o Caso Base em relação a todas as métricas da ANEEL.

In [ ]:
def comparar_cenario_ev_aneel(arquivo_ev_csv, rotulo_cenario):
    """
    Carrega um cenário de penetração de VEs e produz o comparativo das métricas
    da ANEEL (Tensão, DRP, DRC, VUF, FP, Demanda) contra o Caso Base.
    """
    df_ev = pd.read_csv(arquivo_ev_csv)
    df_ev['date'] = pd.to_datetime(df_ev['date'])
    df_ev.set_index('date', inplace=True)
    
    v1_ev = filtrar_tensoes(df_ev, colunas_v1)
    v1_media_ev = v1_ev.mean(axis=1)
    p_ev_kw = df_ev[linha_tronco['p_cols']].sum(axis=1).abs() if linha_tronco['p_cols'] else pd.Series(0, index=df_ev.index)
    
    fig, (ax_v, ax_p) = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
    
    ax_v.plot(df_base.index, v1_media, color="black", linestyle="--", linewidth=1.5, label="Caso Base (0 VEs)")
    ax_v.plot(df_ev.index, v1_media_ev, color="crimson", linewidth=1.8, label=f"Cenário VEs ({rotulo_cenario})")
    ax_v.axhline(0.92, color="orange", linestyle=":", label="Limite Inferior Adequado (0.92 pu)")
    ax_v.set_title(f"Comparativo de Perfil de Tensão Média (Fase A): Caso Base vs. {rotulo_cenario}", fontweight="bold")
    ax_v.set_ylabel("Tensão Média (pu)")
    ax_v.grid(True)
    ax_v.legend(loc="lower right")
    
    ax_p.plot(df_base.index, p_linhas_kw, color="black", linestyle="--", linewidth=1.5, label="Demanda Caso Base (kW)")
    ax_p.plot(df_ev.index, p_ev_kw, color="firebrick", linewidth=1.8, label=f"Demanda com VEs ({rotulo_cenario})")
    ax_p.set_title(f"Impacto na Curva de Demanda Agregada: Caso Base vs. {rotulo_cenario}", fontweight="bold")
    ax_p.set_ylabel("Potência Ativa (kW)")
    ax_p.set_xlabel("Dia e Horário")
    ax_p.grid(True)
    ax_p.xaxis.set_major_formatter(mdates.DateFormatter("%a, %d/%m"))
    plt.xticks(rotation=30)
    ax_p.legend(loc="upper right")
    
    plt.tight_layout()
    plt.show()
    
    delta_pico = p_ev_kw.max() - p_linhas_kw.max()
    delta_vmin = v1_ev.min().min() - df_v1.min().min()
    
    df_comp = pd.DataFrame({
        "Métrica": ["Tensão Mínima Global (pu)", "Tensão Média Global (pu)", "Pico de Demanda (kW)", "Energia Semanal (kWh)"],
        "Caso Base (0 VEs)": [f"{df_v1.min().min():.4f}", f"{v1_media.mean():.4f}", f"{p_linhas_kw.max():.2f}", f"{energia_linhas_kwh:,.1f}"],
        f"{rotulo_cenario}": [f"{v1_ev.min().min():.4f}", f"{v1_media_ev.mean():.4f}", f"{p_ev_kw.max():.2f}", f"{(p_ev_kw * delta_t_h).sum():,.1f}"],
        "Delta (Impacto)": [f"{delta_vmin:+.4f} pu", f"{(v1_media_ev.mean() - v1_media.mean()):+.4f} pu", f"{delta_pico:+.2f} kW ({100.0 * delta_pico / p_linhas_kw.max():+.1f}%)", f"{((p_ev_kw * delta_t_h).sum() - energia_linhas_kwh):+,.1f} kWh"]
    })
    print("=" * 75)
    print(f" IMPACTO COMPARATIVO ANEEL: CASO BASE vs. {rotulo_cenario.upper()}")
    print("=" * 75)
    display(df_comp)

print("Módulo de comparação compilado com sucesso!")